In [1]:
def leer_matriz_3x3(nombre_archivo):
    with open(nombre_archivo, "r", encoding="utf-8") as archivo:
        filas = [
            linea.strip()
            for linea in archivo
            if linea.strip()
        ]

    # Validar que existan exactamente 3 filas
    if len(filas) != 3:
        raise ValueError("El archivo debe contener exactamente 3 filas.")

    # Validar que cada fila tenga 3 dígitos
    if any(len(fila) != 3 or not fila.isdigit() for fila in filas):
        raise ValueError("Cada fila debe contener exactamente 3 dígitos.")

    matriz = [
        [int(digito) for digito in fila]
        for fila in filas
    ]

    return matriz

In [2]:
def generar_movimientos(matriz):
    matrices_resultantes = []

    # Buscar la posición del cero
    fila_cero = None
    columna_cero = None

    for fila in range(len(matriz)):
        for columna in range(len(matriz[fila])):
            if matriz[fila][columna] == 0:
                fila_cero = fila
                columna_cero = columna
                break

        if fila_cero is not None:
            break

    if fila_cero is None:
        raise ValueError("La matriz no contiene el valor 0.")

    # Posibles movimientos en el orden solicitado:
    # izquierda, arriba, derecha, abajo
    movimientos = [
        (0, -1),  # Columna anterior
        (-1, 0),  # Fila anterior
        (0, 1),   # Columna siguiente
        (1, 0)    # Fila siguiente
    ]

    for cambio_fila, cambio_columna in movimientos:
        nueva_fila = fila_cero + cambio_fila
        nueva_columna = columna_cero + cambio_columna

        # Verificar que la nueva posición esté dentro de la matriz
        if (
            0 <= nueva_fila < len(matriz)
            and 0 <= nueva_columna < len(matriz[0])
        ):
            # Crear una copia independiente de la matriz
            nueva_matriz = [fila[:] for fila in matriz]

            # Intercambiar el cero con la casilla correspondiente
            nueva_matriz[fila_cero][columna_cero] = (
                nueva_matriz[nueva_fila][nueva_columna]
            )
            nueva_matriz[nueva_fila][nueva_columna] = 0

            matrices_resultantes.append(nueva_matriz)

    return matrices_resultantes

In [4]:
def calcular_diferencia(matriz_actual, matriz_objetivo):
    """
    Calcula la distancia Manhattan total.

    Para cada ficha del 1 al 8, calcula cuántas filas y columnas
    la separan de su posición en la matriz objetivo.
    El cero no se incluye.
    """

    if (
        len(matriz_actual) != 3
        or len(matriz_objetivo) != 3
        or any(len(fila) != 3 for fila in matriz_actual)
        or any(len(fila) != 3 for fila in matriz_objetivo)
    ):
        raise ValueError("Las dos matrices deben ser de tamaño 3x3.")

    posiciones_objetivo = {}

    # Guardar la posición final de cada ficha
    for fila in range(3):
        for columna in range(3):
            ficha = matriz_objetivo[fila][columna]
            posiciones_objetivo[ficha] = (fila, columna)

    suma_total = 0

    # Calcular distancia de cada ficha
    for fila in range(3):
        for columna in range(3):
            ficha = matriz_actual[fila][columna]

            # El espacio vacío no se incluye
            if ficha != 0:
                fila_objetivo, columna_objetivo = (
                    posiciones_objetivo[ficha]
                )

                distancia = (
                    abs(fila - fila_objetivo)
                    + abs(columna - columna_objetivo)
                )

                suma_total += distancia

    return suma_total

In [5]:
def escoger_mejor_matriz(lista_matrices, matriz_objetivo):
    if not lista_matrices:
        raise ValueError("La lista de matrices está vacía.")

    mejor_matriz = None
    menor_diferencia = float("inf")
    posicion_mejor = None

    for posicion, matriz in enumerate(lista_matrices):
        diferencia = calcular_diferencia(matriz, matriz_objetivo)

        print(
            f"Matriz {posicion + 1}: "
            f"diferencia total = {diferencia}"
        )

        if diferencia < menor_diferencia:
            menor_diferencia = diferencia
            mejor_matriz = matriz
            posicion_mejor = posicion

    return mejor_matriz, menor_diferencia, posicion_mejor

In [6]:
def resolver_puzzle(estado_inicial, estado_final):
    # Copia independiente del estado inicial
    estado_actual = [
        fila[:] for fila in estado_inicial
    ]

    numero_movimientos = 0

    # Guardar el recorrido completo
    historial = [
        [fila[:] for fila in estado_actual]
    ]

    # Guardar los estados ya recorridos
    estados_visitados = [
        [fila[:] for fila in estado_actual]
    ]

    print("Estado inicial:")

    for fila in estado_actual:
        print(fila)

    while estado_actual != estado_final:

        print("\n===================================")
        print("Movimiento número:", numero_movimientos + 1)

        # 1. Generar movimientos posibles
        lista_matrices = generar_movimientos(
            estado_actual
        )

        # 2. Quitar matrices ya visitadas
        matrices_disponibles = []

        for matriz in lista_matrices:
            if matriz not in estados_visitados:
                matrices_disponibles.append(matriz)

        # Evitar continuar si no existen movimientos nuevos
        if not matrices_disponibles:
            print(
                "\nNo se puede continuar porque todos los "
                "movimientos posibles ya fueron visitados."
            )
            break

        # 3. Escoger la mejor matriz
        mejor_matriz, menor_diferencia, posicion_mejor = (
            escoger_mejor_matriz(
                matrices_disponibles,
                estado_final
            )
        )

        # 4. Reemplazar estado_actual
        estado_actual = [
            fila[:] for fila in mejor_matriz
        ]

        numero_movimientos += 1

        # Guardar la matriz recorrida
        estados_visitados.append(
            [fila[:] for fila in estado_actual]
        )

        historial.append(
            [fila[:] for fila in estado_actual]
        )

        print("\nMatriz seleccionada:")

        for fila in estado_actual:
            print(fila)

        print("Menor diferencia:", menor_diferencia)
        print(
            "Posición seleccionada en la lista:",
            posicion_mejor
        )
        print(
            "Movimientos realizados:",
            numero_movimientos
        )

    solucion_encontrada = (
        estado_actual == estado_final
    )

    if solucion_encontrada:
        print("\n¡Se encontró el estado final!")
    else:
        print(
            "\nEl algoritmo se detuvo sin encontrar "
            "el estado final."
        )

    return (
        estado_actual,
        numero_movimientos,
        estados_visitados,
        solucion_encontrada
    )

In [12]:
estadoInicial = leer_matriz_3x3("estadoinicial.txt")
#[
#    [3, 2, 6],
#    [5, 0, 4],
#    [1, 8, 7]
#]

estadoFinal = leer_matriz_3x3("estadofinal.txt")
#[
#    [8, 4, 7],
#    [2, 6, 5],
#    [3, 1, 0]
#]

estadoActual, movimientos, historial, solucion = resolver_puzzle(
    estadoInicial,
    estadoFinal
)
print("\nRECORRIDO COMPLETO")



Estado inicial:
[3, 2, 6]
[5, 0, 4]
[1, 8, 7]

Movimiento número: 1
Matriz 1: diferencia total = 15
Matriz 2: diferencia total = 15
Matriz 3: diferencia total = 15
Matriz 4: diferencia total = 15

Matriz seleccionada:
[3, 2, 6]
[0, 5, 4]
[1, 8, 7]
Menor diferencia: 15
Posición seleccionada en la lista: 0
Movimientos realizados: 1

Movimiento número: 2
Matriz 1: diferencia total = 14
Matriz 2: diferencia total = 16

Matriz seleccionada:
[0, 2, 6]
[3, 5, 4]
[1, 8, 7]
Menor diferencia: 14
Posición seleccionada en la lista: 0
Movimientos realizados: 2

Movimiento número: 3
Matriz 1: diferencia total = 13

Matriz seleccionada:
[2, 0, 6]
[3, 5, 4]
[1, 8, 7]
Menor diferencia: 13
Posición seleccionada en la lista: 0
Movimientos realizados: 3

Movimiento número: 4
Matriz 1: diferencia total = 12
Matriz 2: diferencia total = 14

Matriz seleccionada:
[2, 6, 0]
[3, 5, 4]
[1, 8, 7]
Menor diferencia: 12
Posición seleccionada en la lista: 0
Movimientos realizados: 4

Movimiento número: 5
Matriz 1: di